## 1 Input Parsing

In [12]:
def parse_test_case(nl_text):
    lines = [l.strip() for l in nl_text.split("\n") if l.strip()]
    test_name = lines[0].replace("Goal:", "").strip().lower().replace(" ", "_")

    raw_steps = []
    in_steps = False
    for line in lines:
        lower = line.lower()
        if lower.startswith("steps"):
            in_steps = True
            continue
        if lower.startswith("expected"):
            in_steps = False
            continue
        if in_steps and line[0].isdigit():
            raw_steps.append(line.split(".", 1)[1].strip())

    return {"test_name": test_name, "raw_steps": raw_steps}


## 2 Intermediate Representation

In [13]:
def build_ir(parsed):
    return {
        "test_name": parsed["test_name"],
        "steps": [{"action": "raw", "value": s} for s in parsed["raw_steps"]]
    }


## 3 Application Model Mapping

In [14]:
SELECTOR_MAP = {
    "username": "#username",
    "password": "#password",
    "login_button": "#login-button",
}

def map_ir(ir):
    mapped = []
    for step in ir["steps"]:
        mapped.append(step)  # placeholder
    return {"test_name": ir["test_name"], "steps": mapped}


## 4 Deterministic Code Generation

In [15]:
ACTION_MAP = {
    "navigate": lambda s: f'        await page.goto("{s["value"]}")',
    "enter_text": lambda s: f'        await page.fill("{s["selector"]}", "{s["value"]}")',
    "click": lambda s: f'        await page.click("{s["selector"]}")',
}

def generate_test_code(ir):
    header = [
        "from playwright.async_api import async_playwright, expect",
        "import asyncio",
        "",
        f"async def test_{ir['test_name']}():",
        "    async with async_playwright() as p:",
        "        browser = await p.chromium.launch(headless=False)",
        "        page = await browser.new_page()",
        ""
    ]

    body = []
    for step in ir["steps"]:
        if step["action"] in ACTION_MAP:
            body.append(ACTION_MAP[step["action"]](step))
        else:
            body.append(f"        # Unmapped step: {step['value']}")

    footer = [
        "",
        "        await browser.close()",
        "",
        "if __name__ == '__main__':",
        f"    asyncio.run(test_{ir['test_name']}())"
    ]

    return "\n".join(header + body + footer)


## 5 Execution and Feedback Loop

In [16]:
import subprocess, json, datetime
from pathlib import Path

def run_test_file(path):
    result = subprocess.run(["python", str(path)], capture_output=True, text=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    artifacts = Path("artifacts") / f"{path.stem}_{timestamp}"
    artifacts.mkdir(parents=True, exist_ok=True)

    summary = {
        "file": str(path),
        "return_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "artifacts": str(artifacts)
    }

    with open(artifacts / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary


## 6 NL to NR Rules

In [ ]:
def interpret_step(text):
    t = text.lower()

    if t.startswith("navigate to"):
        url = text.split("to", 1)[1].strip()
        return {"action": "navigate", "value": url}

    if t.startswith("enter username"):
        value = text.split('"')[1]
        return {"action": "enter_text", "selector": "username", "value": value}

    if t.startswith("enter password"):
        value = text.split('"')[1]
        return {"action": "enter_text", "selector": "password", "value": value}

    if t.startswith("click") and "login" in t:
        return {"action": "click", "selector": "login_button"}

    if t.startswith("verify") and "visible" in t:
        logical = text.split("visible")[0].replace("Verify", "").strip()
        return {"action": "assert_visible", "selector": logical}

    return {"action": "raw", "value": text}


## 7 Updated IR Builder

In [ ]:
def build_ir(parsed):
    ir_steps = [interpret_step(s) for s in parsed["raw_steps"]]
    return {"test_name": parsed["test_name"], "steps": ir_steps}


## 8 Selector Mapping Upgrade

In [ ]:
SELECTOR_MAP = {
    "username": "#username",
    "password": "#password",
    "login_button": "#login-button",
    "dashboard_heading": "h1",
}

def map_ir(ir):
    mapped = []

    for step in ir["steps"]:
        new_step = step.copy()

        # Replace logical selector with real CSS selector
        if "selector" in new_step:
            logical = new_step["selector"]
            if logical in SELECTOR_MAP:
                new_step["selector"] = SELECTOR_MAP[logical]

        mapped.append(new_step)

    return {"test_name": ir["test_name"], "steps": mapped}


## 9 Action Mapping Upgrade

In [ ]:
ACTION_MAP = {
    "navigate": lambda s: f'        await page.goto("{s["value"]}")',
    "enter_text": lambda s: f'        await page.fill("{s["selector"]}", "{s["value"]}")',
    "click": lambda s: f'        await page.click("{s["selector"]}")',
    "assert_visible": lambda s: f'        await expect(page.locator("{s["selector"]}")).to_be_visible()',
}


## 10 Execution and Feedback

In [ ]:
import subprocess, json, datetime
from pathlib import Path

def run_test_file(path):
    result = subprocess.run(["python", str(path)], capture_output=True, text=True)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    artifacts = Path("artifacts") / f"{path.stem}_{timestamp}"
    artifacts.mkdir(parents=True, exist_ok=True)

    summary = {
        "file": str(path),
        "return_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "artifacts": str(artifacts)
    }

    with open(artifacts / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary


## 11 Full Pipeline

from pathlib import Path

def run_pipeline(nl_text, output_path="generated_test.py"):
    # 1. Parse NL
    parsed = parse_test_case(nl_text)

    # 2. Build IR
    ir = build_ir(parsed)

    # 3. Map selectors
    mapped = map_ir(ir)

    # 4. Generate Playwright code
    code = generate_test_code(mapped)

    # 5. Write test file
    path = Path(output_path)
    path.write_text(code)

    # 6. Execute + collect artifacts
    return run_test_file(path)


## 12 Example NL test

In [ ]:
example_test = """
Goal: Verify user can log in
URL: http://localhost:5000/login

Steps:
  1. Navigate to /login
  2. Enter username "demo"
  3. Enter password "password123"
  4. Click login button
  5. Verify dashboard_heading visible

Expected Result:
  User is logged in and sees the dashboard.
"""
